# Beginner E3: explore accent structure without training a classifier

This is a step-by-step version of **E3 — Unsupervised learning**. We use the frozen Whisper embeddings saved by the beginner E2 notebook. **K-means** groups recordings by similarity; **PCA** and **UMAP** make those embeddings easier to view in two dimensions.

By the end, you will know how to:

1. load the same speech embeddings used in E2;
2. scale and reduce the embeddings;
3. form two clusters without giving K-means accent labels;
4. visualize the clusters with PCA and UMAP; and
5. compare the clusters with known labels *afterward*.

> Clusters are exploratory patterns in the supplied recordings. They are not accent predictions or evidence of a person's ethnicity, nationality, or identity.


## Before you start

Run `notebooks/E2_01_whisper_classifier_beginner.ipynb` through its **Extract once, then cache** cell first. It writes a Whisper embedding cache to `Singlish/outputs/e2` in Google Drive. Set `MAX_CLIPS_PER_CLASS` to the same value in E2 and E3 so E3 loads the matching cache. You do not need to train the E2 classifier to use this notebook.

Use the same `PROJECT_ROOT` as E2. A small subset of speakers is fine for learning, but tiny datasets make clusters and UMAP plots unstable. This notebook needs at least four usable recordings and embeddings from at least two speakers.


In [ ]:
# Install the libraries used here. The E2 embedding cache avoids rerunning Whisper.
%pip -q install scikit-learn umap-learn matplotlib numpy


## 1. Connect to the project folder

Only change `PROJECT_ROOT` if your Drive folder has a different location. E3 saves its figures and report under `outputs/e3`; it does not change the E2 cache.


In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/Singlish')
MAX_CLIPS_PER_CLASS = None  # None uses all cached audio; match this setting in E2.
if MAX_CLIPS_PER_CLASS is not None and (not isinstance(MAX_CLIPS_PER_CLASS, int) or isinstance(MAX_CLIPS_PER_CLASS, bool) or MAX_CLIPS_PER_CLASS < 1):
    raise ValueError('MAX_CLIPS_PER_CLASS must be None or a positive integer')
cache_suffix = '' if MAX_CLIPS_PER_CLASS is None else f'_max{MAX_CLIPS_PER_CLASS}'
EMBEDDING_CACHE = PROJECT_ROOT / 'outputs' / 'e2' / f'beginner_whisper_embeddings{cache_suffix}.npz'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'e3'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42

print('Embedding cache:', EMBEDDING_CACHE)
print('Saving results to:', OUTPUT_DIR)


In [ ]:
import csv
import json
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import umap
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

if not EMBEDDING_CACHE.is_file():
    raise FileNotFoundError(
        f'Missing {EMBEDDING_CACHE}. Run the E2 notebook through its embedding-cache cell first.'
    )

with np.load(EMBEDDING_CACHE, allow_pickle=False) as cache:
    required = {'embeddings', 'labels', 'groups', 'paths', 'model_name'}
    missing = required - set(cache.files)
    if missing:
        raise ValueError(f'E2 embedding cache is missing: {sorted(missing)}')
    X = cache['embeddings'].astype(np.float32)
    labels = cache['labels'].astype(np.int64)
    groups = cache['groups'].astype(str)
    paths = cache['paths'].astype(str)
    model_name = str(cache['model_name'])

if X.ndim != 2 or X.shape[0] < 4 or X.shape[1] < 2:
    raise ValueError('Need at least four embeddings with at least two values each.')
if not (len(X) == len(labels) == len(groups) == len(paths)):
    raise ValueError('The E2 cache arrays have different numbers of recordings.')
if not np.isfinite(X).all():
    raise ValueError('The embedding cache contains non-finite values.')
if len(set(groups)) < 2:
    raise ValueError('Need recordings from at least two speakers.')
if not set(np.unique(labels)).issubset({0, 1}):
    raise ValueError('Expected E2 labels 0 (non-Singapore) and 1 (Singapore).')

print('Whisper model:', model_name)
print('Embedding matrix:', X.shape, '(recordings × embedding values)')
print('Speakers:', len(set(groups)))
print('Labels available for later interpretation:', Counter(labels))


## 2. Prepare the embeddings

`StandardScaler` puts embedding dimensions on comparable scales. PCA keeps the strongest directions of variation and reduces the number of dimensions before clustering. We use up to 50 components, limited by the dataset size.

Every step below uses **embeddings only**. The `labels` array is deliberately absent from `fit` and `fit_predict` calls. E3 is exploratory, so there is no train/test split or classification score here.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

n_components = min(50, len(X) - 1, X.shape[1])
pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
X_reduced = pca.fit_transform(X_scaled)
print('PCA dimensions used for clustering:', X_reduced.shape[1])
print('Variance retained:', f'{pca.explained_variance_ratio_.sum():.1%}')


## 3. Form two clusters with K-means

The project asks whether the recordings naturally form two groups, so we set `n_clusters=2`. K-means chooses two centers and assigns each recording to its nearest center in the reduced embedding space. Cluster numbers `0` and `1` are arbitrary; neither means “Singapore” by itself.

The silhouette score measures whether points are nearer their own cluster than the other one. It measures **separation**, not agreement with accents. Scores near 1 indicate clearer separation; near 0 indicate overlap. A high score can still reflect microphone or corpus differences.


In [ ]:
kmeans = KMeans(n_clusters=2, n_init=10, random_state=RANDOM_SEED)
clusters = kmeans.fit_predict(X_reduced)
cluster_counts = Counter(clusters)
if len(cluster_counts) == 2 and min(cluster_counts.values()) > 1:
    silhouette = float(silhouette_score(X_reduced, clusters))
else:
    silhouette = None  # A one-recording cluster cannot give a useful silhouette.

print('Cluster sizes:', dict(sorted(cluster_counts.items())))
print('Silhouette score:', 'not available' if silhouette is None else f'{silhouette:.3f}')


## 4. Draw PCA and UMAP maps

PCA's first two components show the two largest linear directions of variation. UMAP makes a second two-dimensional view that may show local neighborhoods more clearly. These maps are for **visualization**: K-means was fitted to the higher-dimensional `X_reduced`, not the 2D UMAP coordinates.

UMAP can give a different-looking layout when settings or data change. Distances between faraway UMAP islands should not be read as exact distances between accents.


In [ ]:
# PCA may have only one component when the dataset is extremely narrow;
# pad the visual with zero so the plot still has two axes.
pca_xy = np.zeros((len(X), 2), dtype=np.float32)
pca_xy[:, :min(2, X_reduced.shape[1])] = X_reduced[:, :2]

umap_xy = umap.UMAP(
    n_components=2,
    n_neighbors=min(15, len(X) - 1),
    min_dist=0.1,
    random_state=RANDOM_SEED,
    init='random',  # Works more reliably on small beginner datasets.
).fit_transform(X_reduced)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, xy, title, names in [
    (axes[0], pca_xy, 'PCA view', ('PC 1', 'PC 2')),
    (axes[1], umap_xy, 'UMAP view', ('UMAP 1', 'UMAP 2')),
]:
    for cluster in (0, 1):
        mask = clusters == cluster
        ax.scatter(xy[mask, 0], xy[mask, 1], s=35, alpha=0.75,
                   label=f'Cluster {cluster} (n={mask.sum()})')
    ax.set(title=title, xlabel=names[0], ylabel=names[1])
    ax.legend()
fig.suptitle('Whisper embedding clusters — labels hidden during clustering')
fig.tight_layout()
cluster_figure = OUTPUT_DIR / 'beginner_e3_clusters.png'
fig.savefig(cluster_figure, dpi=180, bbox_inches='tight')
plt.show()


## 5. Reveal the known accent labels

Only now do we use the E2 labels. A table counts the recordings from each known class inside each cluster. The **adjusted Rand index (ARI)** compares the two groupings: approximately 0 means no better agreement than chance; 1 means perfect agreement. ARI does not turn clustering into a classifier or establish that accent caused the separation.

If every cluster is dominated by one corpus, check recording conditions, speaker balance, clip length, and prompts before interpreting it as accent structure. Several clips from one speaker can also crowd the plot and affect the result.


In [ ]:
class_names = {0: 'Non-Singapore', 1: 'Singapore'}
print('Known accent labels inside each cluster:')
print('Cluster | Non-Singapore | Singapore')
for cluster in (0, 1):
    counts = Counter(labels[clusters == cluster])
    print(f'{cluster:^7} | {counts[0]:^13} | {counts[1]:^9}')

ari = float(adjusted_rand_score(labels, clusters))
print(f'Adjusted Rand index: {ari:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, xy, title in [(axes[0], pca_xy, 'PCA view'), (axes[1], umap_xy, 'UMAP view')]:
    for label in (0, 1):
        mask = labels == label
        ax.scatter(xy[mask, 0], xy[mask, 1], s=35, alpha=0.75,
                   label=f'{class_names[label]} (n={mask.sum()})')
    ax.set(title=title, xlabel='Dimension 1', ylabel='Dimension 2')
    ax.legend()
fig.suptitle('Known labels shown after clustering')
fig.tight_layout()
label_figure = OUTPUT_DIR / 'beginner_e3_known_labels.png'
fig.savefig(label_figure, dpi=180, bbox_inches='tight')
plt.show()


## 6. Save the result

The assignment CSV keeps each recording, speaker ID, cluster, and known label together for later inspection. The report stores the settings and summary scores. These are saved to Google Drive, alongside the two figures.


In [ ]:
assignments_path = OUTPUT_DIR / 'beginner_e3_assignments.csv'
with assignments_path.open('w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['path', 'speaker_group', 'cluster', 'known_label'])
    for path, group, cluster, label in zip(paths, groups, clusters, labels):
        writer.writerow([path, group, int(cluster), class_names[int(label)]])

report = {
    'experiment': 'E3 unsupervised embedding exploration',
    'embedding_model': model_name,
    'recordings': int(len(X)),
    'speakers': int(len(set(groups))),
    'embedding_dimensions': int(X.shape[1]),
    'pca_dimensions_for_clustering': int(n_components),
    'pca_variance_retained': float(pca.explained_variance_ratio_.sum()),
    'clusters': {str(k): int(v) for k, v in sorted(cluster_counts.items())},
    'silhouette_score': silhouette,
    'adjusted_rand_index_after_clustering': ari,
    'known_label_counts_by_cluster': {
        str(cluster): {
            class_names[label]: int(np.sum((clusters == cluster) & (labels == label)))
            for label in (0, 1)
        }
        for cluster in (0, 1)
    },
    'random_seed': RANDOM_SEED,
    'caution': 'Clusters may reflect corpus, speaker, recording, or prompt differences rather than accent.',
}
report_path = OUTPUT_DIR / 'beginner_e3_report.json'
report_path.write_text(json.dumps(report, indent=2) + '\n')

print('Saved:')
for path in (cluster_figure, label_figure, assignments_path, report_path):
    print(' -', path)


## What to try next

1. Check whether clips from one speaker cluster together. If so, repeat the analysis with one embedding per speaker (average that speaker's clips) and compare.
2. Repeat with a better-balanced set of speakers and recording conditions from both corpora.
3. Try a different cluster count only if you have a clear exploration question; record the choice before interpreting labels.
4. In the full project, keep E3 as exploratory evidence alongside the supervised E1/E2 evaluation. Do not report ARI or silhouette as held-out classification accuracy.
